In [1]:
import random
import os
import sys
from pathlib import Path
import contextlib
import render_generator_v2
import endpoint_generator_v2
import reduction_engine
from schemas import (
  validate_object_library,
  validate_assembly,
  v2_structural_gate
)
import segment_analysis
import gwl_serializer
import itertools
import multiprocessing
import pathos.multiprocessing
from functools import partial

In [2]:
def generate_mushroom_entry(
  stem_d=3.0, 
  stem_h=6.0, 
  cap_d=6.0, 
  cap_h=2.0, 
  grid_size=None, 
  spacing=15.0
):
  if grid_size is None: grid_size = (10,10)
  """
  Generates a mushroom (nailhead) metamaterial design.
  Ensures centers are calculated relative to the ground (z=0).
  """
  
  # Calculate centers so they sit perfectly on top of each other
  stem_center_z = stem_h / 2
  cap_center_z = stem_h + (cap_h / 2)
  
  # Procedural Job Name
  job_id = f"mush_s{stem_d}_c{cap_d}_p{spacing}"

  data = {
    "job_name": job_id,
    "prompt": f"Create a {grid_size[0]}x{grid_size[1]} array of mushroom resonators. "
          f"The stem should be {stem_h}um tall with a diameter of {stem_d}um. "
          f"The top cap should be {cap_d}um wide and {cap_h}um thick. "
          f"Space them {spacing}um apart.",
    "objects": {
      "mushroom": {
        "type": "geometry",
        "components": [
          {
            "type": "cylinder", 
            "center": [0, 0, round(stem_center_z, 3)], 
            "dimensions": {"diameter_um": stem_d, "height_um": stem_h}
          },
          {
            "type": "cylinder", 
            "center": [0, 0, round(cap_center_z, 3)], 
            "dimensions": {"diameter_um": cap_d, "height_um": cap_h}
          }
        ]
      }
    },
    "assembly": {
      "type": "grid",
      "grid": {"x": grid_size[0], "y": grid_size[1]},
      "spacing_um": {"x": spacing, "y": spacing},
      "default_object": "mushroom"
    }
  }
  return data

In [3]:
def generate_gammadion_entry(
  center_w=2.0,  # Width of the central cross bars
  center_l=6.0,  # Total length of the central cross bars
  arm_l=3.0,     # Length of the chiral "bent" part of the arms
  arm_w=1.5,     # Width of the chiral arms
  thickness=1.0,   # Height (z-dimension) of the whole structure
  spacing=15.0,
  grid_size=None,
  handedness="right" # "right" or "left" determines the chirality
):
  if grid_size is None: grid_size = (10,10)
  """
  Generates a Chiral Gammadion using axis-aligned boxes.
  Consists of: 1 central cross (2 boxes) and 4 peripheral arms (4 boxes).
  """
  
  # Handedness multiplier for arm offsets
  h_mult = 1 if handedness == "right" else -1
  
  # Components list
  components = []
  
  # 1. Central Cross (Two intersecting bars)
  components.append({
    "type": "box", "center": [0, 0, thickness/2], 
    "dimensions": {"width_um": center_l, "depth_um": center_w, "height_um": thickness}
  })
  components.append({
    "type": "box", "center": [0, 0, thickness/2], 
    "dimensions": {"width_um": center_w, "depth_um": center_l, "height_um": thickness}
  })
  
  # 2. Four Chiral Arms (placed at the ends of the cross)
  # The arms are offset based on the center_l and handedness
  offset = center_l / 2
  arm_offset = (arm_l / 2) * h_mult
  
  # North Arm
  components.append({
    "type": "box", "center": [arm_offset, offset, thickness/2],
    "dimensions": {"width_um": arm_l, "depth_um": arm_w, "height_um": thickness}
  })
  # South Arm
  components.append({
    "type": "box", "center": [-arm_offset, -offset, thickness/2],
    "dimensions": {"width_um": arm_l, "depth_um": arm_w, "height_um": thickness}
  })
  # East Arm
  components.append({
    "type": "box", "center": [offset, -arm_offset, thickness/2],
    "dimensions": {"width_um": arm_w, "depth_um": arm_l, "height_um": thickness}
  })
  # West Arm
  components.append({
    "type": "box", "center": [-offset, arm_offset, thickness/2],
    "dimensions": {"width_um": arm_w, "depth_um": arm_l, "height_um": thickness}
  })

  return {
    "job_name": f"gammadion_{handedness}_{center_l}um",
    "prompt": f"Design a {handedness}-handed chiral gammadion array. "
          f"The central cross has a length of {center_l}um. "
          f"Each chiral arm is {arm_l}um long. Total thickness is {thickness}um.",
    "objects": {"gammadion": {"type": "geometry", "components": components}},
    "assembly": {
      "type": "grid", "grid": {"x": grid_size[0], "y": grid_size[1]}, 
      "spacing_um": {"x": spacing, "y": spacing},
      "default_object": "gammadion"
    }
  }

In [4]:
def generate_srr_entry(
  outer_dim=5.0,   # Total width/depth of the square
  width=1.0,     # Width of the metal traces/bars
  gap_size=1.0,  # The size of the split in the ring
  thickness=0.5,   # Height (z-dimension)
  spacing=10.0,
  grid_size=None
):
  if grid_size is None: grid_size = (10,10)
  """
  Generates a single-split square resonator using axis-aligned boxes.
  Decomposed into 3 bars: Top, Bottom, and Left (the 'C' shape).
  The gap is on the Right side.
  """
  
  components = []
  z_pos = thickness / 2
  
  # 1. Top Bar
  components.append({
    "type": "box", 
    "center": [0, outer_dim/2 - width/2, z_pos],
    "dimensions": {"width_um": outer_dim, "depth_um": width, "height_um": thickness}
  })
  
  # 2. Bottom Bar
  components.append({
    "type": "box", 
    "center": [0, -(outer_dim/2 - width/2), z_pos],
    "dimensions": {"width_um": outer_dim, "depth_um": width, "height_um": thickness}
  })
  
  # 3. Left Bar (Connecting Top and Bottom)
  # Length is outer_dim - 2*width to avoid overlapping corners double-counting volume
  inner_len = outer_dim - (2 * width)
  components.append({
    "type": "box", 
    "center": [-(outer_dim/2 - width/2), 0, z_pos],
    "dimensions": {"width_um": width, "depth_um": inner_len, "height_um": thickness}
  })

  # Note: For a true SRR, researchers often add two small 'stubs' on the right 
  # to define a precise gap. We'll stick to the 3-bar "C" for simplicity.

  return {
    "job_name": f"srr_gap{gap_size}_w{width}",
    "prompt": f"Create a square split-ring resonator (SRR) array. "
          f"The ring has an outer dimension of {outer_dim}um and a trace width of {width}um. "
          f"The split (gap) is on the right side. Space units {spacing}um apart.",
    "objects": {"srr": {"type": "geometry", "components": components}},
    "assembly": {
      "type": "grid", "grid": {"x": grid_size[0], "y": grid_size[1]},
      "spacing_um": {"x": spacing, "y": spacing},
      "default_object": "srr"
    }
  }

In [5]:
def generate_dimer_entry(
  radius_um=1.0, 
  height_um=2.0, 
  gap_um=0.2, 
  shape_type="cylinder", # "cylinder" or "box"
  grid_size=(10, 10), 
  spacing_um=15.0
):
  """
  Generates a plasmonic dimer (two particles).
  The gap is centered at x=0, so the particles are at +/- (radius + gap/2).
  """
  
  # Calculate horizontal offset from center to center
  # For a box, 'radius' would be half-width. 
  x_offset = radius_um + (gap_um / 2)
  
  # Define dimensions based on shape
  dims = {"diameter_um": radius_um * 2, "height_um": height_um} if shape_type == "cylinder" else \
       {"width_um": radius_um * 2, "depth_um": radius_um * 2, "height_um": height_um}

  components = [
    {"type": shape_type, "center": [-round(x_offset, 3), 0, height_um/2], "dimensions": dims},
    {"type": shape_type, "center": [round(x_offset, 3), 0, height_um/2], "dimensions": dims}
  ]

  return {
    "job_name": f"dimer_{shape_type}_g{gap_um}",
    "prompt": f"Construct an array of {shape_type} dimers. Each particle has a "
          f"{'diameter' if shape_type=='cylinder' else 'width'} of {radius_um*2}um. "
          f"The critical gap between the two particles is {gap_um}um. "
          f"Arrange in a {grid_size[0]}x{grid_size[1]} grid.",
    "objects": {"dimer_unit": {"type": "geometry", "components": components}},
    "assembly": {
      "type": "grid", 
      "grid": {"x": grid_size[0], "y": grid_size[1]},
      "spacing_um": {"x": spacing_um, "y": spacing_um},
      "default_object": "dimer_unit"
    }
  }

In [6]:
def generate_cactus_entry(
    trunk_h=10.0, 
    trunk_d=1.0, 
    spine_l=3.0, 
    spine_d=0.4, 
    levels=3, # Number of vertical "floors" of spines
    grid_size=None,
    spacing=20.0
):
    if grid_size is None: grid_size = (10,10)
    
    components = []
    
    # 1. The Main Trunk (Vertical Cylinder)
    components.append({
        "type": "cylinder", 
        "center": [0, 0, trunk_h/2], 
        # FIXED: Changed diameter_um from trunk_h to trunk_d
        "dimensions": {"diameter_um": trunk_d, "height_um": trunk_h} 
    })
    
    # 2. The Spines (Hierarchical Branching)
    level_heights = [trunk_h * (i+1)/(levels+1) for i in range(levels)]
    spine_offset = (trunk_d / 2) + (spine_l / 2) # Center of spine relative to trunk center
    
    for z in level_heights:
        # X-direction spines 
        # FIXED: Changed to 'box' type because cylinders are strictly Z-aligned
        # FIXED: Mapped dimensions strictly to x_um, y_um, z_um
        components.append({
            "type": "box", "center": [spine_offset, 0, z],
            "dimensions": {"x_um": spine_l, "y_um": spine_d, "z_um": spine_d} 
        })
        components.append({
            "type": "box", "center": [-spine_offset, 0, z],
            "dimensions": {"x_um": spine_l, "y_um": spine_d, "z_um": spine_d}
        })
        
        # Y-direction spines
        # FIXED: Mapped dimensions strictly to x_um, y_um, z_um
        components.append({
            "type": "box", "center": [0, spine_offset, z],
            "dimensions": {"x_um": spine_d, "y_um": spine_l, "z_um": spine_d}
        })
        components.append({
            "type": "box", "center": [0, -spine_offset, z],
            "dimensions": {"x_um": spine_d, "y_um": spine_l, "z_um": spine_d}
        })

    return {
        "job_name": f"cactus_h{trunk_h}_l{levels}",
        "prompt": f"Design a 3D cactus meta-atom array. Features a central trunk {trunk_h}um tall "
                  f"with {levels} levels of orthogonal spines. Spines are {spine_l}um long.",
        "objects": {"cactus": {"type": "geometry", "components": components}},
        "assembly": {
            "type": "grid", "grid": {"x": grid_size[0], "y": grid_size[1]},
            "spacing_um": {"x": spacing, "y": spacing},
            "default_object": "cactus"
        }
    }

In [14]:
import json
import random

# --- [Insert the 5 Factory Functions here: Mushroom, Gammadion, SRR, Dimer, Cactus] ---

def generate_design_dataset(samples_per_type=200):
  dataset = []

  # 1. LOOP: MUSHROOMS (Vertical Stacking / Surface Waves)
  for _ in range(samples_per_type):
    s_d = round(random.uniform(1.0, 4.0), 2)
    # Constraint: Cap must be 1.5x to 3x wider than stem
    c_d = round(s_d * random.uniform(1.5, 3.0), 2)
    pitch = round(c_d + random.uniform(2.0, 8.0), 2)
    dataset.append(generate_mushroom_entry(stem_d=s_d, cap_d=c_d, spacing=pitch))

  # 2. LOOP: CHIRAL GAMMADIONS (Polarization / Chirality)
  for _ in range(samples_per_type):
    c_l = round(random.uniform(4.0, 8.0), 2)
    a_l = round(c_l * random.uniform(0.3, 0.6), 2)
    hand = random.choice(["right", "left"])
    pitch = round(c_l + a_l + 5.0, 2)
    dataset.append(generate_gammadion_entry(center_l=c_l, arm_l=a_l, handedness=hand, spacing=pitch))

  # # 3. LOOP: SPLIT-RING RESONATORS (Negative Permeability / Gaps)
  # for _ in range(samples_per_type):
  #   o_dim = round(random.uniform(5.0, 12.0), 2)
  #   w = round(o_dim * 0.15, 2) # Maintain proportional trace width
  #   gap = round(random.uniform(0.5, 2.0), 2)
  #   pitch = round(o_dim + 5.0, 2)
  #   dataset.append(generate_srr_entry(outer_dim=o_dim, width=w, gap_size=gap, spacing=pitch))

  # # 4. LOOP: DIMERS (Plasmonic Hotspots)
  # for _ in range(samples_per_type):
  #   r = round(random.uniform(0.5, 2.5), 2)
  #   g = round(random.uniform(0.1, 0.8), 2) # Sub-micron gaps are "real"
  #   h = round(r * 3, 2)
  #   shape = random.choice(["cylinder", "box"])
  #   dataset.append(generate_dimer_entry(radius_um=r, height_um=h, gap_um=g, shape_type=shape))

  # 5. LOOP: 3D CACTI (Hierarchical 3D Branching)
  for _ in range(samples_per_type):
    h = round(random.uniform(10.0, 25.0), 2)
    lev = random.randint(2, 5)
    s_len = round(h * 0.25, 2)
    pitch = round((s_len * 2) + 10.0, 2)
    dataset.append(generate_cactus_entry(trunk_h=h, levels=lev, spine_l=s_len, spacing=pitch))


  # Validate dataset - fail immediately on any error.
  for design in dataset:
    v2_structural_gate(design)
    object_library_errors = validate_object_library({"objects": design.get("objects", {})})
    if len(object_library_errors) > 1:
      raise ValueError(f"Object library for design {design['job_name']} contained errors: {object_library_errors}\n\nDesign:\n{design}")
    object_names = list(design.get("objects", {}).keys())
    assembly_errors = validate_assembly({"assembly": design.get("assembly", {})}, object_names)
    if len(assembly_errors) > 1:
      raise ValueError(f"Assembly for design {design['job_name']} contained errors: {object_library_errors}\n\nDesign:\n{design}")
    
  

  return dataset

In [15]:
# Generate design dataset
metamaterials_dataset = generate_design_dataset(samples_per_type=5)
print(f"Success! Generated {len(metamaterials_dataset)} unique design/prompt pairs.")

[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 structural gate on keys: ['job_name', 'prompt', 'objects', 'assembly']
[GATE] Running v2 str

In [16]:
def process_design(design: dict, print_params: dict, gwl_params: dict, dataset_dir = "Dataset"):
  output_dir = Path(dataset_dir, design['job_name'])
  if not output_dir.is_dir():
    output_dir.mkdir()

  # Dump design
  with open(output_dir / "design.json", "w") as f:
    json.dump(design, f, indent=2)

  # Generate Renders
  render_generator_v2.generate_object_aware_renders(design, Path(output_dir, "Renders"))

  # Reduce primitives
  reduced = reduction_engine.reduce_assembly(design)
  reduction_errors = reduction_engine.validate_reduced_output(reduced)
  if len(reduction_errors) > 1:
    raise ValueError(f"Validation error reducting primitives for design {design['job_name']}: {object_library_errors}\n\nDesign:\n{design}")
  
  with open(output_dir / "reduced.json", "w") as f:
    json.dump(reduced, f, indent=2)

  # Generate endpoints
  endpoint_data = endpoint_generator_v2.generate_endpoint_json_v2(reduced, print_params)
  with open(output_dir / "output.json", 'w') as f:
    json.dump(endpoint_data, f, indent=2)

  # Generate GWL
  gwl_dir = output_dir / "GWL"
  gwl_files = gwl_serializer.generate_gwl_files(endpoint_data, gwl_params, gwl_dir)
  master_gwl = gwl_dir / f"{design['job_name']}_master.gwl"
  gwl_serializer.generate_master_gwl(gwl_files, gwl_params, master_gwl)

  # Run segment analysis
  segment_analysis.analyze_segments(print_params, gwl_dir, output_dir / "segments_analyzed.json")

  return output_dir

In [17]:
def mute_process():
  sys.stdout = open(os.devnull, 'w')

# Run pipeline for analysis
if __name__ == "__main__":
  if not Path("Dataset").is_dir():
    Path("Dataset").mkdir()
  
  # Distribute processing efficiently across cores
  with pathos.multiprocessing.Pool(initializer = mute_process) as pool:
    count = 0
    print_params = endpoint_generator_v2.load_print_parameters("PrintParameters.txt")
    gwl_params = gwl_serializer.load_gwl_parameters("PrintParameters.txt")
    p_func = partial(process_design, dataset_dir = "Dataset", print_params = print_params, gwl_params = gwl_params)
    for result in pool.imap_unordered(p_func, metamaterials_dataset, chunksize=1):
      count = count + 1
      print(result, f"({count} / {len(metamaterials_dataset)} items")

Dataset/gammadion_left_6.75um (1 / 15 items
Dataset/gammadion_left_7.72um (2 / 15 items
Dataset/gammadion_right_5.92um (3 / 15 items
Dataset/gammadion_right_4.29um (4 / 15 items
Dataset/gammadion_right_4.05um (5 / 15 items
Dataset/mush_s1.05_c1.63_p5.39 (6 / 15 items
Dataset/mush_s2.6_c4.0_p7.71 (7 / 15 items
Dataset/cactus_h10.44_l2 (8 / 15 items
Dataset/mush_s3.02_c7.5_p14.66 (9 / 15 items
Dataset/mush_s3.33_c8.34_p10.66 (10 / 15 items
Dataset/mush_s3.77_c8.03_p14.17 (11 / 15 items
Dataset/cactus_h21.56_l2 (12 / 15 items
Dataset/cactus_h15.47_l5 (13 / 15 items
Dataset/cactus_h21.29_l3 (14 / 15 items
Dataset/cactus_h17.73_l4 (15 / 15 items


# Qdrant Dataset Generation

In [18]:
from qdrant_client import QdrantClient # 1.17.0
from qdrant_client.models import PointStruct, VectorParams, Datatype, Distance
from langchain_ollama import OllamaEmbeddings
import json

In [19]:
embeddings_model = OllamaEmbeddings(model="mxbai-embed-large", base_url="http://127.0.0.1:11434")

In [20]:
points = []
for listing in os.listdir(Path("Dataset")):
  project_dir = Path("Dataset", listing)
  with open(project_dir / "design.json", 'r') as f:
    design = json.load(f)
  with open(project_dir / "segments_analyzed.json", 'r') as f:
    segments_analyzed = json.load(f)
  
  embedding = embeddings_model.embed_query(design["prompt"])  # Compute embedding

  # We lack good error analysis right now - guess vaguely for the model.
  if segments_analyzed["failed_segments"] == 0:
    problems = []
  else:
    problems = ["Contains traces that are not adhered (likely due to a floating object or overhang)"]
        
  points.append(
    PointStruct(
      id=hash(design["prompt"]),  # Unique ID for the point
      vector=embedding,  # The embedding vector
      payload={"prompt": design["prompt"], "design": design, "printable": segments_analyzed["failed_segments"] == 0, "problems": problems}
    )
  )

In [21]:
# Create qdrant vector store
qclient = QdrantClient(path = './Dataset/qdrant_dataset')
qclient.create_collection(
    collection_name="prompts",
    vectors_config=VectorParams(
        size=1024,
        distance=Distance.COSINE,
        datatype=Datatype.FLOAT32
    )
)

True

In [22]:
# Upload points
qclient.upsert(collection_name="prompts", points=points)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

# Example querying

In [41]:
with open("prompt.txt", 'r') as f:
  embedding = embeddings_model.embed_query(f.read())
search_results = qclient.query_points(
    collection_name="prompts",
    query=embedding,
    with_payload=True,
    limit=8
).points

In [47]:
successful_example = None
failed_example = None
# Try to pick one successful example and one failed example
for result in search_results:
  if result.payload["printable"] == True:
    successful_example = result.payload
    break

for result in search_results:
  if result.payload["printable"] == False:
    failed_example = result.payload
    break

In [48]:
successful_example

{'prompt': 'Construct an array of cylinder dimers. Each particle has a diameter of 2.18um. The critical gap between the two particles is 0.71um. Arrange in a 10x10 grid.',
 'design': {'job_name': 'dimer_cylinder_g0.71',
  'prompt': 'Construct an array of cylinder dimers. Each particle has a diameter of 2.18um. The critical gap between the two particles is 0.71um. Arrange in a 10x10 grid.',
  'objects': {'dimer_unit': {'type': 'geometry',
    'components': [{'type': 'cylinder',
      'center': [-1.445, 0, 1.635],
      'dimensions': {'diameter_um': 2.18, 'height_um': 3.27}},
     {'type': 'cylinder',
      'center': [1.445, 0, 1.635],
      'dimensions': {'diameter_um': 2.18, 'height_um': 3.27}}]}},
  'assembly': {'type': 'grid',
   'grid': {'x': 10, 'y': 10},
   'spacing_um': {'x': 15.0, 'y': 15.0},
   'default_object': 'dimer_unit'}},
 'printable': True,
 'problems': []}

In [49]:
failed_example

{'prompt': 'Grad- Create an array of nailhead-style structures. Each structure consists of two stacked cylinders: a base cylinder that is 6 um tall and 3 um in diameter, and a top cylinder centered on the base that is 2 um tall and 6 um in diameter. Arranged in a 10x10 square array with 15 um spacing.',
 'design': {'metadata': {'category': '',
   'timestamp': '20260215_013906',
   'model': 'qwen2.5-coder:3b',
   'tokens': {'prompt_tokens': 1287,
    'completion_tokens': 326,
    'total_tokens': 1613},
   'errors': []},
  'prompt': 'Grad- Create an array of nailhead-style structures. Each structure consists of two stacked cylinders: a base cylinder that is 6 um tall and 3 um in diameter, and a top cylinder centered on the base that is 2 um tall and 6 um in diameter. Arranged in a 10x10 square array with 15 um spacing.',
  'job_name': 'nailheads_array',
  'objects': {'base_nailhead_cylinder': {'type': 'geometry',
    'description': 'Base nailhead cylinder',
    'components': [{'type': 'c

# Evaluating The Model Against The Dataset